# Hospedando Agentes LlamaIndex com modelos Amazon Bedrock no Amazon Bedrock AgentCore Runtime com Observabilidade

## Visão Geral

Neste tutorial, aprenderemos como hospedar seu agente LlamaIndex usando o Amazon Bedrock AgentCore Runtime com observabilidade integrada. Este tutorial demonstra como implantar um agente LlamaIndex no AgentCore Runtime e capturar automaticamente dados de telemetria para monitoramento e análise.

### Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo do tutorial    | Conversacional                                                                   |
| Tipo de agente      | Único                                                                            |
| Framework de agente | LlamaIndex                                                                       |
| Modelo LLM          | Anthropic Claude Haiku                                                           |
| Componentes do tutorial | Hospedagem de agente no AgentCore Runtime com Observabilidade               |
| Vertical do tutorial | Cross-vertical                                                                  |
| Complexidade do exemplo | Fácil                                                                        |
| SDK utilizado        | Amazon BedrockAgentCore Python SDK e boto3                                      |

### Arquitetura do Tutorial

Neste tutorial, descreveremos como implantar um agente LlamaIndex no AgentCore Runtime com observabilidade automática.

Para fins de demonstração, usaremos um FunctionAgent do LlamaIndex com modelos Amazon Bedrock e ferramentas aritméticas.

### Principais Funcionalidades do Tutorial

* Hospedagem de Agentes LlamaIndex no Amazon Bedrock AgentCore Runtime
* Uso de modelos Amazon Bedrock
* Observabilidade e rastreamento automáticos
* Coleta de telemetria integrada

## Pré-requisitos

Para executar este tutorial, você precisará de:
* Python 3.10+
* Credenciais AWS
* Amazon Bedrock AgentCore SDK
* LlamaIndex
* Acesso ao modelo Amazon Bedrock Claude Haiku

### No Seu Terminal:

`cd 01-tutorials/06-AgentCore-observability/01-Agentcore-runtime-hosted/LlamaIndex`

`python -m venv venv`

`source venv/bin/activate`

### No Seu Notebook:

Selecione seu venv como kernel

In [ ]:
!pip install -q --force-reinstall -U -r requirements.txt

## Criando seu agente LlamaIndex e experimentando localmente

Antes de implantar nosso agente no AgentCore Runtime, vamos desenvolvê-lo e executá-lo localmente para fins de experimentação.

Para aplicações de agentes em produção, precisaremos desacoplar o processo de criação do agente do processo de invocação. Com o AgentCore Runtime, decoraremos a parte de invocação do nosso agente com o decorador `@app.entrypoint` e o teremos como ponto de entrada para nosso runtime.

In [ ]:
%%writefile llamaindex_agent.py

import warnings
warnings.filterwarnings("ignore", message=".*validate_default.*", category=UserWarning)
import os
import json
import argparse
import boto3
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.observability.otel import LlamaIndexOpenTelemetry


# Initialize OpenTelemetry instrumentation for LlamaIndex
instrumentor = LlamaIndexOpenTelemetry()
# Start listening
instrumentor.start_registering()

def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b

def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b

def get_bedrock_model():
    model_id = "anthropic.claude-3-5-haiku-20241022-v1:0"
    region = boto3.Session().region_name
    
    bedrock_model = BedrockConverse(
        model=model_id,
        region_name=region,
    )
    return bedrock_model

# Initialize the model
bedrock_model = get_bedrock_model()

# Create the arithmetic agent
agent = FunctionAgent(
    tools=[add, multiply],
    llm=bedrock_model,
)

async def llamaindex_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    response = await agent.run(user_input)
    return str(response)

if __name__ == "__main__":
    import asyncio
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = asyncio.run(llamaindex_agent_bedrock(json.loads(args.payload)))
    print(response)

#### Invocando o agente local

In [ ]:
!python llamaindex_agent.py '{"prompt": "What is (121 + 2) * 5?"}'

## Preparando seu agente para implantação no AgentCore Runtime

Agora vamos implantar nosso agente no AgentCore Runtime. Para isso, precisamos:
* Importar o Runtime App com `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Inicializar o App em nosso código com `app = BedrockAgentCoreApp()`
* Decorar a função de invocação com o decorador `@app.entrypoint`
* Permitir que o AgentCoreRuntime controle a execução do agente com `app.run()`

### Agente LlamaIndex com modelo Amazon Bedrock
Vamos preparar nosso Agente LlamaIndex para implantação no AgentCore Runtime.

In [ ]:
%%writefile llamaindex_agent.py

import warnings
warnings.filterwarnings("ignore", message=".*validate_default.*", category=UserWarning)
import os
import json
import boto3
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.observability.otel import LlamaIndexOpenTelemetry


app = BedrockAgentCoreApp()

# Initialize OpenTelemetry instrumentation for LlamaIndex
instrumentor = LlamaIndexOpenTelemetry(debug=True)
# Start listening
instrumentor.start_registering()

def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b

def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b

def get_bedrock_model():
    model_id = "anthropic.claude-3-5-haiku-20241022-v1:0"
    region = boto3.Session().region_name
    
    bedrock_model = BedrockConverse(
        model=model_id,
        region_name=region,
    )
    return bedrock_model

# Initialize the model
bedrock_model = get_bedrock_model()

# Create the arithmetic agent
agent = FunctionAgent(
    tools=[add, multiply],
    llm=bedrock_model,
)

@app.entrypoint
async def llamaindex_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = await agent.run(user_input)
    return str(response)

if __name__ == "__main__":
    app.run()

## O que acontece nos bastidores?

Quando você usa o `BedrockAgentCoreApp`, ele automaticamente:

* Cria um servidor HTTP que escuta na porta 8080
* Implementa o endpoint `/invocations` necessário para processar os requisitos do agente
* Implementa o endpoint `/ping` para verificações de saúde
* Gerencia tipos de conteúdo e formatos de resposta adequados
* Gerencia o tratamento de erros de acordo com os padrões AWS
* **Habilita automaticamente a observabilidade e coleta de telemetria**

## Implantando o agente no AgentCore Runtime

A operação `CreateAgentRuntime` suporta opções abrangentes de configuração, permitindo especificar imagens de contêiner, variáveis de ambiente e configurações de criptografia. Você também pode configurar protocolos (HTTP, MCP) e mecanismos de autorização para controlar como seus clientes se comunicam com o agente.

**Nota:** A melhor prática operacional é empacotar o código como contêiner e enviar para o ECR usando pipelines CI/CD e IaC

Neste tutorial, usaremos o Amazon Bedrock AgentCore Python SDK para empacotar facilmente seus artefatos e implantá-los no AgentCore Runtime.

### Configurar a implantação do AgentCore Runtime

Primeiro, usaremos nosso starter toolkit para configurar a implantação do AgentCore Runtime com um entrypoint, a execution role que acabamos de criar e um arquivo de requisitos. Também configuraremos o starter kit para criar automaticamente o repositório Amazon ECR no lançamento.

Durante a etapa de configuração, seu docker file será gerado com base no código da sua aplicação

In [ ]:
from bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "llamaindex_bedrock_getting_started10"
response = agentcore_runtime.configure(
    entrypoint="llamaindex_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)
response

### Lançando o agente no AgentCore Runtime

Agora que temos um docker file, vamos lançar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime com observabilidade habilitada automaticamente. Você pode adicionar bibliotecas à variável de ambiente abaixo para excluir traces desnecessários de bibliotecas específicas.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        # Minimal set - only disable truly noisy instrumentations
        "OTEL_PYTHON_DISABLED_INSTRUMENTATIONS": (
            "jinja2,"
            "urllib3,"
            "requests,"
            "httpx,"
            "redis,"
            "aiohttp-client"
            # Add "starlette" to this list to get rid of the POST /invocations trace. Note: this will disable session tracking.
        )
    }
)


### Verificando o Status do AgentCore Runtime
Agora que implantamos o AgentCore Runtime, vamos verificar o status da implantação

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

## Habilitar Rastreamento para seu AgentCore Runtime

No console AWS, navegue até o Amazon Bedrock AgentCore. Clique em Agent Runtime em Build and Deploy e selecione seu agente.

Para seu agente, role até ver a seção Tracing. Habilite-a para permitir a entrega de traces ao CloudWatch:

![enable_tracing.png](images/llamaindex_enable_tracing.png)

### Invocando o AgentCore Runtime

Finalmente, podemos invocar nosso AgentCore Runtime com um payload. Isso gerará automaticamente dados de telemetria que podem ser visualizados no painel de observabilidade.

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What is (121 + 2) * 5?"})
invoke_response

### Processando resultados da invocação

Agora podemos processar nossos resultados de invocação para incluí-los em uma aplicação

In [ ]:
from IPython.display import Markdown, display
import json
response_text = invoke_response['response'][0]
display(Markdown(response_text))

### Invocando o AgentCore Runtime com boto3

Agora que seu AgentCore Runtime foi criado, você pode invocá-lo com qualquer AWS SDK. Por exemplo, você pode usar o método `invoke_agent_runtime` do boto3.

In [ ]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 15 * 8?"})
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## Painel de Observabilidade

### Coleta Automática de Telemetria

Quando seu agente LlamaIndex é executado no AgentCore Runtime, dados de telemetria são coletados automaticamente e enviados ao Amazon CloudWatch. Isso inclui:

- **Traces de execução do agente**: Fluxo completo do processo de tomada de decisão do seu agente
- **Chamadas LLM**: Invocações de modelos Bedrock com tokens de entrada/saída
- **Uso de ferramentas**: Chamadas de funções e seus resultados
- **Métricas de desempenho**: Latência, uso de tokens e taxas de erro

### Visualizando Traces no CloudWatch

Para visualizar os dados de observabilidade do seu agente:

1. Navegue até o console do AWS CloudWatch
2. Vá para o painel **GenAI Observability**
3. Selecione seu agent runtime para visualizar traces e métricas

### Principais Funcionalidades de Observabilidade

- **Rastreamento de sessões**: Correlacione múltiplas interações
- **Monitoramento de erros**: Identifique e depure problemas
- **Análise de desempenho**: Otimize os tempos de resposta do agente

### Observabilidade do AgentCore no Amazon CloudWatch

Em resumo, siga os passos abaixo para habilitar a observabilidade de agentes hospedados no AgentCore Runtime:

- Habilite o Transaction Search no Amazon CloudWatch
- O arquivo requirements.txt contém `aws-opentelemetry-distro` listado ao implantar o agente no Bedrock AgentCore Runtime.

## Visão Geral do Bedrock AgentCore no painel GenAI Observability

Você pode visualizar todos os seus Agentes que possuem observabilidade e filtrar os dados com base em intervalos de tempo. Alguns exemplos são fornecidos abaixo:

![genai-observability.png](images/llamaindex_dashboard_view.png)

No painel principal, você pode visualizar métricas de runtime de todos os agentes, conforme mostrado abaixo:

![runtime-all-agent-metrics.png](images/llamaindex_runtime_metrics_total_view.png)

Agora, se você clicar no agente que acabou de implantar, será direcionado a um painel com as métricas de runtime específicas deste agente. Você também pode filtrar os dados por um intervalo de tempo personalizado:

![runtime-metrics-per-agent.png](images/llamaindex_runtime_metrics_view.png)

Na aba Sessions View, você pode navegar por todas as sessões associadas a este agente:

![Agent-sessions-view.png](images/llamaindex_sessions_view.png)

Na aba Trace View, você pode examinar os traces e informações de span deste agente no runtime:

![Agentcore-trace.png](images/llamaindex_traces_view.png)

Navegue pelas diversas funcionalidades do painel GenAI Observability para obter informações mais detalhadas sobre os traces.


## Limpeza (Opcional)

Vamos agora limpar o AgentCore Runtime criado

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
import boto3
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
    
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

# Parabéns!

Você concluiu com sucesso:

- Criação de um agente LlamaIndex com ferramentas aritméticas
- Implantação no Amazon Bedrock AgentCore Runtime
- Habilitação de observabilidade automática e coleta de telemetria
- Invocação do agente e geração de dados de trace

Seu agente agora está em execução com recursos completos de observabilidade, permitindo que você monitore o desempenho, depure problemas e otimize suas aplicações de agentes.